# Huấn luyện v3 trên Google Colab (GPU) — họ hàm mất mát trung vị + đối chứng

Notebook này chỉ là **trình chạy mỏng** (CLAUDE.md mục 10: code thật nằm trong module `.py`). Nó giải nén gói code + dataset do `ml/scripts/package_for_colab.py` đóng,
kiểm môi trường, rồi gọi `ml/scripts/train_models_v3.py`. Không cần MySQL/backend.

**Cách dùng:** tải **đúng 3 file của MỘT thư mục** gói v3 (`outputs/colab/<dataset_version>_v3_r3/`: `ml_train_pkg_*.zip`, `dataset_<dataset_version>.zip`, `COLAB_MANIFEST.json`) lên **một thư mục Drive riêng cho từng dataset**
(`DRIVE_DIR`). **Không trộn hai gói trong cùng một thư mục** (hai `COLAB_MANIFEST.json` sẽ đè nhau). Dataset single-horizon: dev1b = h1, dev3b = h3 — chạy notebook này **hai lần**, mỗi lần với `DRIVE_DIR` và `DATASET_VERSION` của dataset đó.

**Môi trường FAIL-CLOSED (hợp đồng C9'):** phiên bản scikit-learn/xgboost/numpy/pandas phải khớp bản đã ghim trong `configs/train_v3.yaml`. Ô 3 kiểm và, nếu lệch, cài đúng phiên bản rồi **yêu cầu Restart runtime** — bạn chạy lại từ ô 1.
`ALLOW_ENV_DRIFT = True` chỉ để thử khi bất khả kháng: kết quả ghi `exploratory_env_drift` (không dùng chung identity với run đúng môi trường).

**Kết quả luôn là `dev_research_exploratory`** (dataset `purpose=dev`, test dev đã lộ): dùng để phát triển, KHÔNG phải số liệu cuối của luận văn. Runtime → Change runtime type → GPU (T4/L4/A100); XGBoost dùng GPU, HGB/RF/Ridge chạy CPU.

In [ ]:
# 1) Tham số — sửa các dòng này
DRIVE_DIR = "/content/drive/MyDrive/hotel_price_intelligence_v3_r3"   # thư mục Drive chứa đúng 3 file của MỘT gói v3 (2 file zip + COLAB_MANIFEST.json)
DATASET_VERSION = "ds_XXXXXXXX_xxx"                                  # vd ds_20261006_dev1b (h1) hoặc ds_20261006_dev3b (h3); khớp tên file dataset_<...>.zip
HORIZONS = ""                                                        # để TRỐNG = tự lấy đúng horizon được đánh giá của dataset (khuyến nghị)
DEVICE = "auto"                                                      # auto: thử cuda, lỗi thì cpu và ghi fallback_reason; hoặc "cuda" / "cpu"
ALLOW_ENV_DRIFT = False                                              # mặc định False: lệch phiên bản => dừng. Chỉ True nếu bất khả kháng (ghi exploratory_env_drift)

from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi -L

In [ ]:
# 2) Kiểm toàn vẹn (SHA-256 so với COLAB_MANIFEST.json) rồi giải nén SẠCH vào /content (xóa bản cũ trước: không để file code/dataset cũ sót lại; CLI còn kiểm tập file == CODE_MANIFEST)
import glob, hashlib, json, os, shutil, zipfile

shutil.rmtree("/content/ml", ignore_errors=True)
shutil.rmtree(f"/content/datasets/{DATASET_VERSION}", ignore_errors=True)
manifest = json.load(open(f"{DRIVE_DIR}/COLAB_MANIFEST.json", encoding="utf-8"))
for name, info in manifest["archives"].items():
    path = f"{DRIVE_DIR}/{name}"
    digest = hashlib.sha256(open(path, "rb").read()).hexdigest()
    assert digest == info["sha256"], f"SHA-256 lệch: {name}"
    zipfile.ZipFile(path).extractall("/content" if name.startswith("ml_train_pkg") else "/content/datasets")
    print("OK", name, f"{info['bytes'] / 1e6:.1f} MB")
assert os.path.exists(f"/content/datasets/{DATASET_VERSION}/samples.parquet"), "không thấy samples.parquet — kiểm DATASET_VERSION"
assert os.path.exists("/content/ml/scripts/train_models_v3.py"), "gói code không phải gói v3 — dùng gói v3-r3 trong thư mục *_v3_r3"
print(glob.glob("/content/ml/*"))

In [ ]:
# 3) Môi trường: so phiên bản với bản ĐÃ GHIM (configs/train_v3.yaml). Lệch => cài đúng phiên bản rồi PHẢI Restart runtime (Runtime -> Restart session) và chạy lại từ ô 1.
import importlib.metadata as md
import yaml

expected = yaml.safe_load(open("/content/ml/configs/train_v3.yaml", encoding="utf-8"))["environment"]["expected_versions"]


def _version(package):
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return None


actual = {package: _version(package) for package in expected}
mismatch = {package: (expected[package], actual[package]) for package in expected if actual[package] != expected[package]}
print("đã ghim:", expected)
print("hiện có:", actual)
if mismatch and not ALLOW_ENV_DRIFT:
    pins = " ".join(f"{package}=={version}" for package, version in expected.items())
    print("LỆCH phiên bản:", mismatch, "-> đang cài đúng phiên bản đã ghim...")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pins.split()], check=True)
    raise RuntimeError("Đã cài đúng phiên bản. BẮT BUỘC: Runtime -> Restart session, rồi chạy lại từ ô 1 (không cài lại giữa các lần fit).")
elif mismatch:
    print("ALLOW_ENV_DRIFT=True: sẽ chạy với môi trường hiện có, kết quả ghi exploratory_env_drift (không official, không chung identity).")
else:
    print("OK: môi trường khớp bản đã ghim.")

In [ ]:
# 4) Huấn luyện + đánh giá (TEST chỉ chạm sau khi khóa champion/routing, chỉ cho danh sách prespecify; thư mục run mới, không ghi đè)
import time

OUT_ROOT = f"{DRIVE_DIR}/models_v3"
RUN_ID = time.strftime("run_%Y%m%d_%H%M%S")
HORIZONS_FLAG = f"--horizons {HORIZONS}" if str(HORIZONS).strip() else ""      # trống => CLI chỉ chạy horizon trong whitelist của dataset
DRIFT_FLAG = "--allow-env-drift" if ALLOW_ENV_DRIFT else ""
!python /content/ml/scripts/train_models_v3.py --dataset-dir /content/datasets/{DATASET_VERSION} {HORIZONS_FLAG} --device {DEVICE} --output-root {OUT_ROOT} --run-id {RUN_ID} --colab-manifest {DRIVE_DIR}/COLAB_MANIFEST.json {DRIFT_FLAG}

In [ ]:
# 5) Tóm tắt (đọc h{k}_report.json của run PASS vừa chạy; run lỗi/dở dang bị bỏ qua). Lift so với persistence; Acc@20% luôn kèm persistence (bão hòa ~94-96%).
import json, pathlib, sys
import pandas as pd

sys.path.insert(0, "/content/ml")
from training.run_transaction import verify_run_dir

root = pathlib.Path(OUT_ROOT, DATASET_VERSION)
run_dir = root / RUN_ID
problems = verify_run_dir(run_dir)
assert not problems, f"run không PASS hợp lệ: {problems} (xem các thư mục *.failed-* nếu có)"
rows, details = [], {}
for path in sorted(run_dir.glob("h*_report.json")):
    r = json.loads(path.read_text(encoding="utf-8"))
    details[r["horizon"]] = r
    if r["status"] != "ok":
        rows.append({"horizon": r["horizon"], "status": r["status"], "reason": r.get("reason")})
        continue
    test = r["test"]["table"]
    val = r["validation_table"]
    for name in test:
        v = val.get(name, {}).get("validation", {}) if name in val else {}
        t = test[name]
        rows.append({"horizon": r["horizon"], "model": name, "validation_lift_vnd": v.get("lift_vnd"), "test_lift_vnd": t["lift_vnd"], "test_lift_log": t["lift_log"],
                     "test_ci95_lift_vnd": t.get("lift_vnd_ci95_hotel"), "test_acc20": round(t["accuracy20"], 4), "persistence_acc20": round(t["persistence_accuracy20"], 4),
                     "tie_share": round(t["share_tie"], 3)})
summary_df = pd.DataFrame(rows)
for r in details.values():
    if r["status"] == "ok":
        print(f"h{r['horizon']}: champion_A={r['champions']['A']['winner']} champion_B={r['champions']['B']['winner']} contract_complete={r['contract_complete']} "
              f"fits={r['ledger']['total_fits']} elapsed={r['ledger']['elapsed_seconds']}s device={r['device']['device']} actual={r['device'].get('actual_device')} verified={(r['device'].get('verification') or {}).get('complete')} fallback={r['device']['fallback_reason']} env={r['environment']['status'] if r.get('environment') else None}")
        print("  cảnh báo:", r["warnings"] or "không", "| claim_level:", r["claim_level"])
summary_df

In [ ]:
# 6) Chi tiết để review: (a) ứng viên mỗi họ theo CV (hai thang), (b) ablation khối tỷ lệ, (c) chính sách routing theo mode (từ VALIDATION), (d) null hoán vị, (e) cảnh báo phạm vi ordinary.
for h, r in details.items():
    if r["status"] != "ok":
        continue
    print(f"=== h{h} ===")
    cand_rows = []
    for family, e in r["families"].items():
        for c in e["candidates"]:
            p = c["pooled"] or {}
            cand_rows.append({"family": family, "family_status": e["status"], "candidate": c["candidate_id"], "status": c["status"], "cv_lift_vnd": p.get("lift_vnd"), "cv_lift_log": p.get("lift_log")})
    display(pd.DataFrame(cand_rows)) if "display" in globals() else print(pd.DataFrame(cand_rows))
    print("ablation:", {k: r["ablation"].get(k) for k in ("status", "reason")}, "| quyết định:", (r["ablation"].get("decision") or {}).get("accepted"))
    print("routing:", {n: {m: (p["use_model"], p["reason"]) for m, p in pol.items()} for n, pol in r["routing"]["policy_by_model"].items()})
    nt = r["null_test"]
    print("null:", {k: nt.get(k) for k in ("status", "reason")}, {k: (nt.get(k) or {}).get("real_gt_max_null") for k in ("lift_vnd", "lift_log")} if nt.get("status") == "ok" else "")
    print("cảnh báo ordinary:", json.dumps(r["claims"]["ordinary_warning"], ensure_ascii=False)[:600])